
# 🔁 Daily Repo Loop for **Ego4d-LiteSTA** (Colab-first)

This block sets up SSH to GitHub (**Deploy Key stored in Drive**), pulls your repo, runs code, logs outputs, and pushes logs/artifacts back to GitHub.
- One-time: add your **Deploy Key** public key to your GitHub repo Deploy Keys with **write access**.
- Edit the variables (**GITHUB_USER**, **REPO_NAME**) below once; they persist per notebook.


In [ ]:
# ⛳ Mount Drive and set variables
from google.colab import drive
drive.mount('/content/drive')

GITHUB_USER = "your-gh-username"          # <<< EDIT ME
REPO_NAME   = "Ego4d-LiteSTA"             # <<< default as requested
REPO_SSH    = f"git@github.com:{GITHUB_USER}/{REPO_NAME}.git"
WORKDIR     = f"/content/{REPO_NAME}"
SSH_DIR     = "/content/drive/MyDrive/.ssh"

print("Repo:", REPO_SSH)
print("Workdir:", WORKDIR)

In [ ]:
# 🔐 SSH + git config + clone/pull
# 1) Git identity (optional but recommended)
!git config --global user.name "Your Name"
!git config --global user.email "your@email.com"

# 2) SSH agent and key from Drive (key filename: colab_egolite)
%%bash -s "$SSH_DIR"
SSH_DIR="$1"
mkdir -p "$SSH_DIR"
chmod 700 "$SSH_DIR"
eval "$(ssh-agent -s)"
ssh-add "$SSH_DIR/colab_egolite" >/dev/null 2>&1 || true
mkdir -p ~/.ssh && chmod 700 ~/.ssh
cat <<EOF > ~/.ssh/config
Host github.com
  HostName github.com
  User git
  IdentityFile $SSH_DIR/colab_egolite
  IdentitiesOnly yes
StrictHostKeyChecking accept-new
EOF
chmod 600 ~/.ssh/config
ssh-keyscan -t rsa,ed25519 github.com >> ~/.ssh/known_hosts 2>/dev/null || true
chmod 644 ~/.ssh/known_hosts

# 3) Clone or pull
import os
if not os.path.exists(WORKDIR):
    !git clone "$REPO_SSH" "$WORKDIR"
else:
    %cd "$WORKDIR"
    !git pull --rebase
%cd "$WORKDIR"
!mkdir -p runs logs outputs notebooks
print("Repo ready at", WORKDIR)

In [ ]:
# 🧾 Run ID + logger helpers (use these in your train/eval cells)
import os, json, traceback, datetime, shutil, glob

RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_DIR = "logs"; RUN_DIR = f"runs/{RUN_ID}"; OUT_DIR = "outputs"
os.makedirs(LOG_DIR, exist_ok=True); os.makedirs(RUN_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

LOG_PATH = f"{LOG_DIR}/{RUN_ID}.log"
ERR_FILE = "COLAB_ERRORS.md"

def log_write(msg):
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(msg + "\n")
    print(msg)

def record_error(title, err_text):
    with open(ERR_FILE, "a", encoding="utf-8") as f:
        f.write(f"\n## {RUN_ID} — {title}\n\n```\n{err_text}\n```\n")

log_write(f"New run: {RUN_ID}")


# 🧪 Ego4D STA — Colab Starter (Data Loading + Sanity Checks)

This notebook is designed to **run on Colab Free** with minimal setup. It focuses on:
- Mounting Google Drive and pointing to your **Ego4D STA** data (annotations + frames).
- Installing minimal dependencies and cloning the **Ego4D Forecasting** repo for official STA metrics.
- Loading a sample from the STA annotations and **visualizing the last frame** with a GT box overlay (val split).
- (Optional) Running the official **evaluation script** on a tiny subset to verify your setup.

> **Tip:** Keep runs short and save outputs to Drive frequently (Colab sessions can disconnect).


In [ ]:

#@title 🔍 Check GPU
import torch, platform, sys
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))


In [ ]:

#@title 🔗 Mount Google Drive (recommended)
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")



## 📁 Set your base paths

Two scenarios:

- **A) You already have STA data in Drive** → set `EGO4D_DIR` to where `annotations/` and `frames/` live.
- **B) You want to download using the Ego4D CLI** (requires credentials) → see the optional download cell below.


In [ ]:

#@title 🔧 Configure paths (edit these to your layout)
from pathlib import Path

# Example layout in Drive (EDIT to your own folder)
EGO4D_DIR = Path("/content/drive/MyDrive/ego4d_data")  #@param {type:"string"}
USE_V2 = True  #@param {type:"boolean"}
SPLIT_DIR = "v2" if USE_V2 else "v1"

# Expected files (you can change these if your file names differ)
STA_TRAIN_JSON = EGO4D_DIR / SPLIT_DIR / "annotations" / "fho_sta_train.json"
STA_VAL_JSON   = EGO4D_DIR / SPLIT_DIR / "annotations" / "fho_sta_val.json"
STA_TEST_JSON  = EGO4D_DIR / SPLIT_DIR / "annotations" / "fho_sta_test_unannotated.json"
METADATA_JSON  = EGO4D_DIR / SPLIT_DIR / "ego4d.json"

# Frames root (high-res last-frame images + low-res clip frames)
FRAMES_DIR = EGO4D_DIR / SPLIT_DIR / "frames"  # e.g., frames/<video_uid>/<frame>.jpg

print("EGO4D_DIR  :", EGO4D_DIR)
print("SPLIT_DIR  :", SPLIT_DIR)
for p in [STA_TRAIN_JSON, STA_VAL_JSON, STA_TEST_JSON, METADATA_JSON, FRAMES_DIR]:
    print("Path      :", p, "| exists:", p.exists())



### ⬇️ (Optional) Download annotations/frames with Ego4D CLI
Only if you have **Ego4D credentials** and want to fetch a **small subset**. Otherwise skip.


In [ ]:

#@title (Optional) Install Ego4D CLI and fetch STA annotations only
DO_DOWNLOAD = False  #@param {type:"boolean"}
if DO_DOWNLOAD:
    !pip -q install ego4d awscli
    import os, getpass
    print("You must have received AWS keys from Ego4D to proceed.")
    aws_key = getpass.getpass("AWS ACCESS KEY: ")
    aws_secret = getpass.getpass("AWS SECRET KEY: ")
    os.environ["AWS_ACCESS_KEY_ID"] = aws_key
    os.environ["AWS_SECRET_ACCESS_KEY"] = aws_secret

    out_dir = EGO4D_DIR
    out_dir.mkdir(parents=True, exist_ok=True)
    # Fetch annotations for forecasting (FHO)
    !ego4d --output_directory="{EGO4D_DIR}" --datasets annotations --benchmarks FHO -y
    print("Downloaded annotations to:", EGO4D_DIR)



## 📦 Install minimal dependencies
We keep it light to stay within Colab Free limits.


In [ ]:

#@title Install libraries
!pip -q install timm==0.9.16 opencv-python==4.10.0.84 matplotlib==3.9.2



## 📚 Get official STA metrics (Ego4D Forecasting repo)


In [ ]:

#@title Clone the Forecasting repo (metrics & references)
%cd /content
!git clone -q https://github.com/EGO4D/forecasting
%cd forecasting
!ls -la | head -n 5



## 🧰 Load STA annotations and metadata
This cell parses the JSONs and prints a sample entry.


In [ ]:

#@title Load annotations & metadata
import json, random
from pathlib import Path

def read_json(p):
    with open(p, "r") as f:
        return json.load(f)

assert STA_VAL_JSON.exists(), f"Missing {STA_VAL_JSON}"
sta_val = read_json(STA_VAL_JSON)
sta_train = read_json(STA_TRAIN_JSON) if STA_TRAIN_JSON.exists() else {"annotations":[]}
ego4d_meta = read_json(METADATA_JSON) if METADATA_JSON.exists() else {}

def pick_sample(sta):
    anns = sta.get("annotations") or sta.get("results") or []
    if not anns:
        return None
    for a in anns:
        if a.get("clip_uid") and (a.get("video_id") or a.get("video_uid")):
            objs = a.get("objects") or a.get("targets") or []
            if objs:
                return a
    return random.choice(anns) if anns else None

sample = pick_sample(sta_val) or pick_sample(sta_train)
print("Entries — train:", len(sta_train.get("annotations",[])), "val:", len(sta_val.get("annotations",[])))
print("Picked sample keys:", list(sample.keys()) if sample else None)
if sample:
    short = {k: sample[k] for k in sample.keys() if k in ["clip_uid","video_id","video_uid","timestamp","objects"]}
    print(json.dumps(short, indent=2)[:800])



## 👀 Visualize a validation sample
- Finds the **last frame** path if your `FRAMES_DIR` follows `<video_uid>/*.jpg` (adjust if your naming differs).
- Draws **one** GT box (if present) for sanity.


In [ ]:

#@title Show last frame + one GT box (val)
import os, glob, cv2, numpy as np, matplotlib.pyplot as plt

def find_last_frame_path(video_uid: str, frames_root: Path):
    folder = frames_root / video_uid
    if not folder.exists():
        return None
    jpgs = sorted(glob.glob(str(folder / "*.jpg")))
    pngs = sorted(glob.glob(str(folder / "*.png")))
    imgs = jpgs if jpgs else pngs
    return imgs[-1] if imgs else None

def draw_box(img, box, thick=2):
    x1, y1, x2, y2 = map(int, box)
    import cv2
    cv2.rectangle(img, (x1,y1), (x2,y2), (255,255,255), thickness=thick)

ann = sample
if ann is None:
    raise SystemExit("No sample available. Check your annotation files.")

video_uid = ann.get("video_id") or ann.get("video_uid")
last_frame = find_last_frame_path(video_uid, FRAMES_DIR)

print("video_uid:", video_uid)
print("Last frame path:", last_frame, "exists:", os.path.exists(last_frame) if last_frame else None)

if last_frame and os.path.exists(last_frame):
    img = cv2.imread(last_frame)[:, :, ::-1]
    H, W = img.shape[:2]
    objs = ann.get("objects", [])
    box = None
    for o in objs:
        if "box" in o:
            box = o["box"]
            break
    if box is not None:
        draw_box(img, box, thick=3)
    plt.figure(figsize=(7,7))
    plt.title(f"{video_uid} — last frame ({W}x{H})")
    plt.axis("off")
    plt.imshow(img)
    plt.show()
else:
    print("Could not locate last frame. Adjust FRAMES_DIR or naming pattern.")



## ✅ (Optional) Mini evaluation scaffold
If you already have a **predictions JSON** in the official format, you can run the repo’s evaluator.  
For now, we’ll create a tiny *dummy* prediction dict aligned to the val split, just to test the path.


In [ ]:

#@title Build a dummy prediction file (format only) and run evaluator
import json

val_anns = sta_val.get("annotations", [])
dummy = {"results": {}}
for a in val_anns[:50]:  # tiny subset
    clip_uid = a.get("clip_uid","unknown")
    dummy["results"][clip_uid] = []

pred_path = "/content/dummy_val_preds.json"
with open(pred_path, "w") as f:
    json.dump(dummy, f)
print("Wrote dummy file:", pred_path)
print("Next: follow forecasting/SHORT_TERM_ANTICIPATION.md to run the official evaluation on your preds.")



## ▶️ Next steps (after data works)
1. Add a lightweight backbone (e.g., **MViTv2-Tiny/Small** or **VideoMAE-S**) and a **simple STA head** that predicts **bbox + verb + noun + TTC** on the last frame.
2. (Optional) Add **hand heatmaps**/**hotspot prior** as extra channels for the last-frame head.
3. (Optional) Implement **training-free token pruning** (temporal static merge + rollout-guided keep set) to cut compute and measure **FPS vs. mAP**.
4. Always save checkpoints to **Drive** and keep runs short.


In [ ]:
# 🚀 Commit & push logs/artifacts
import json, glob, datetime
%cd "$WORKDIR"

# Optional: include metrics summary in the commit message if exists
metrics_candidates = sorted(glob.glob("runs/*/metrics.json"))
metrics_note = ""
if metrics_candidates:
    try:
        with open(metrics_candidates[-1]) as f:
            m = json.load(f)
        metrics_note = f" | sel_acc={m.get('selector_acc')} verb_acc={m.get('verb_acc')} ttc_mae={m.get('ttc_mae')}"
    except Exception as e:
        print("Could not read metrics.json:", e)

!git add logs/* runs/* outputs/* COLAB_ERRORS.md || true
!git status --porcelain

COMMIT_MSG = f"colab: logs & outputs {datetime.datetime.now().isoformat(timespec='seconds')}{metrics_note}"
!git commit -m "$COMMIT_MSG" || echo "Nothing to commit."
!git push
print("Pushed to GitHub.")